# STEP 03: Feature Engineering

This notebook derives rich predictive domain features from property attributes:
1. Floor level features (`Current_Floor`, `Total_Floors`, `Floor_Ratio`, `Is_Top_Floor`, `Is_Ground_Floor`).
2. Property space & layout ratios (`Bathroom_BHK_Ratio`, `Size_Per_BHK`, `Size_Per_Bathroom`).
3. Size category discretization and log size transform (`Size_Category`, `Log_Size`).
4. High-signal interaction categorical features (`City_Locality`, `City_BHK`, `City_Furnishing`).
5. Exporting `engineered_house_rent_dataset.csv`.

In [1]:
# Load cleaned dataset from Step 02
import pandas as pd
import numpy as np
import os

df = pd.read_csv("../dataset/processed/cleaned_house_rent_dataset.csv")
print("Cleaned Dataset Shape:", df.shape)

Cleaned Dataset Shape: (4177, 13)


In [2]:
# Feature Engineering Step 1: Parse raw Floor string into numeric Current_Floor and Total_Floors
def extract_floor_features(value):
    value = str(value).strip()
    if value.lower().startswith("ground"):
        current_floor = 0
    elif value.lower().startswith("upper basement"):
        current_floor = -2
    elif value.lower().startswith("lower basement"):
        current_floor = -1
    else:
        try:
            current_floor = int(value.split()[0])
        except:
            current_floor = 0
    try:
        total_floors = int(value.split("out of")[1].strip())
    except:
        total_floors = 0
    return current_floor, total_floors

if "Floor" in df.columns:
    floors = df["Floor"].apply(extract_floor_features)
    df["Current_Floor"] = [f[0] for f in floors]
    df["Total_Floors"] = [f[1] for f in floors]
    df = df.drop(columns=["Floor"])

df["Current_Floor"] = df.get("Current_Floor", 0).fillna(0)
df["Total_Floors"] = df.get("Total_Floors", 0).fillna(0)
print("Floor features extracted successfully.")

Floor features extracted successfully.


In [3]:
# Feature Engineering Step 2: Compute space, room, and floor position ratios
# Floor_Ratio: relative height of unit in building
df["Floor_Ratio"] = (df["Current_Floor"] / df["Total_Floors"].replace(0, np.nan)).fillna(0)
# Bathroom_BHK_Ratio: ratio of bathrooms to bedrooms
df["Bathroom_BHK_Ratio"] = (df["Bathroom"] / df["BHK"].replace(0, np.nan)).fillna(0)
# Size_Per_BHK: average sqft size per bedroom
df["Size_Per_BHK"] = (df["Size"] / df["BHK"].replace(0, np.nan)).fillna(0)
# Size_Per_Bathroom: average sqft size per bathroom
df["Size_Per_Bathroom"] = (df["Size"] / df["Bathroom"].replace(0, np.nan)).fillna(0)
# Binary indicators for top floor penthouse and ground floor unit
df["Is_Top_Floor"] = ((df["Current_Floor"] == df["Total_Floors"]) & (df["Total_Floors"] > 0)).astype(int)
df["Is_Ground_Floor"] = (df["Current_Floor"] == 0).astype(int)

In [4]:
# Feature Engineering Step 3: Categorize property size and log-transform numerical Size
def size_category(size):
    if size < 800:
        return "Small"
    elif size < 1500:
        return "Medium"
    elif size < 2500:
        return "Large"
    else:
        return "Very Large"

df["Size_Category"] = df["Size"].apply(size_category)
df["Log_Size"] = np.log1p(df["Size"])

# Feature Engineering Step 4: Create high-signal interaction features
# City_Locality: neighborhood-specific target encoding
df["City_Locality"] = df["City"].astype(str) + "_" + df["Area Locality"].astype(str)
# City_BHK: city-level bedroom configuration interaction
df["City_BHK"] = df["City"].astype(str) + "_" + df["BHK"].astype(str) + "BHK"
# City_Furnishing: city-level furnishing status interaction
df["City_Furnishing"] = df["City"].astype(str) + "_" + df["Furnishing Status"].astype(str)

In [6]:
# Export engineered dataset for model training
os.makedirs("../dataset/processed", exist_ok=True)
df.to_csv("../dataset/processed/engineered_house_rent_dataset.csv", index=False)
print("Engineered dataset saved. Shape:", df.shape)
print("Engineered Columns:", df.columns.tolist())

Engineered dataset saved. Shape: (4177, 25)
Engineered Columns: ['BHK', 'Rent', 'Size', 'Area Type', 'Area Locality', 'City', 'Furnishing Status', 'Tenant Preferred', 'Bathroom', 'Posted_Year', 'Posted_Month', 'Posted_DayOfWeek', 'Current_Floor', 'Total_Floors', 'Floor_Ratio', 'Bathroom_BHK_Ratio', 'Size_Per_BHK', 'Size_Per_Bathroom', 'Is_Top_Floor', 'Is_Ground_Floor', 'Size_Category', 'Log_Size', 'City_Locality', 'City_BHK', 'City_Furnishing']
